The cells below until the part marked SVM is just the pre-processing pipeline

In [1]:
# Finds every audio clip: capuchin calls (positives) and other sounds (negatives).

from glob import glob

CAPUCHIN_FILES_PATHS = glob(r"data/Parsed_Capuchinbird_Clips/*.wav")
NON_CAPUCHINE_FILE_PATHS = glob(r"data/Parsed_Not_Capuchinbird_Clips/*.wav")

print(f"Number of Capuchin files: {len(CAPUCHIN_FILES_PATHS)}")
print(f"Number of Non-Capuchin files: {len(NON_CAPUCHINE_FILE_PATHS)}")

Number of Capuchin files: 217
Number of Non-Capuchin files: 593


In [2]:
# Shuffles the negative clips and keeps only as many as there are capuchin clips,
# so the model sees a 50/50 mix of "call" and "not call".

import random

# We are shuffling because the noise files are currently order by their types in the directory
# random.shuffle(NON_CAPUCHINE_FILE_PATHS)

# Minor change, ignore the random.shuffle thing above for now
# Sort first so the file order doesn't depend on how glob lists them, then shuffle with a
# fixed seed so the same 217 negatives (and the same validation split) are picked every run.
CAPUCHIN_FILES_PATHS = sorted(CAPUCHIN_FILES_PATHS)
NON_CAPUCHINE_FILE_PATHS = sorted(NON_CAPUCHINE_FILE_PATHS)
random.Random(0).shuffle(NON_CAPUCHINE_FILE_PATHS)

NON_CAPUCHINE_FILE_PATHS = NON_CAPUCHINE_FILE_PATHS[:len(CAPUCHIN_FILES_PATHS)]

print(f"Number of Capuchin files: {len(CAPUCHIN_FILES_PATHS)}")
print(f"Number of Non-Capuchin files: {len(NON_CAPUCHINE_FILE_PATHS)}")

Number of Capuchin files: 217
Number of Non-Capuchin files: 217


In [3]:
# Reads the shared pipeline settings from config.yaml (sample rate, window length, spectrogram settings) so this notebook processes audio exactly like the real pipeline.

import yaml

spec_cfg = None

with open(r"../configs/config.yaml", "r") as f:
  config = yaml.safe_load(f)

  try:
      spec_cfg = config["spec_config"]
      shared = config["shared"]
  except KeyError as e:
      raise KeyError(
          f"config.yaml is missing expected key {e}. Check that 'shared.sr' and 'spec_config' are present."
      ) from e
FILTER_ORDER = spec_cfg["filter_order"]
N_FFT = spec_cfg["n_fft"]
HOP_LENGTH = spec_cfg["hop_length"]
N_MELS = spec_cfg["n_mels"]

WINDOW_SIZE = shared["window_size"]
SR = shared["sr"]

print(f"FILTER_ORDER = {FILTER_ORDER}")
print(f"N_FFT = {N_FFT}")
print(f"HOP_LENGTH = {HOP_LENGTH}")
print(f"N_MELS = {N_MELS}")

print(f"WINDOW_SIZE = {WINDOW_SIZE}")
print(f"SR = {SR}")

FILTER_ORDER = 4
N_FFT = 512
HOP_LENGTH = 256
N_MELS = 64
WINDOW_SIZE = 3.0
SR = 24000


# SVM

#### SVM made using sklearn

In [4]:
# 1. Loads every clip and cuts a 3-second window from it, as the device would.
# 2. Turns each window into a spectrogram, then summarises it as 100 numbers describing the sound's shape and how it changes over time.
# 3. Splits the data by recording, so clips from the same recording never appear in both training and validation (otherwise the score would be inflated).
# 4. Optionally makes 8 degraded copies of each training clip (noise, muffling, volume changes) so the model copes with lower-quality microphones.

import sys, os, re, random
from pathlib import Path
import numpy as np
import soundfile as sf
from scipy.fft import dct
 
sys.path.insert(0, str(Path.cwd().parent))
from features.mel_spectogram_generator import MelSpectrogramGen
from training.data_prep.audio_normalisation import AudioNorm
from sklearn.metrics import average_precision_score, roc_auc_score
 
SEED = 0
random.seed(SEED); np.random.seed(SEED)
 
mel = MelSpectrogramGen.from_config()
norm = AudioNorm(window_size=WINDOW_SIZE, sr=SR, hop_size=2.0, clip_pad_max_ratio=0.25)
 
files  = list(CAPUCHIN_FILES_PATHS) + list(NON_CAPUCHINE_FILE_PATHS)
labels = np.array([1] * len(CAPUCHIN_FILES_PATHS) + [0] * len(NON_CAPUCHINE_FILE_PATHS))
 
def load_mono(path):
    y, sr = sf.read(path, dtype="float32", always_2d=True)
    return np.ascontiguousarray(y.mean(axis=1)), sr

cache = [load_mono(f) for f in files]

N_MFCC = 20

# Each clip becomes 100 numbers that summarises how the sound behaves over time.
def clip_features(logmel):
    """(n_mels, n_frames) log-mel dB -> 100-dim feature vector."""
    M = logmel - logmel.mean(axis=1, keepdims=True) # Subtrtact each band's mean over time. 
 
    # This boils each moment down to 20 numbers that describe the overall shape of the sound.
    C = dct(M, type=2, axis=0, norm="ortho")[1:1 + N_MFCC]        
    d = np.diff(C, axis=1) # how fast that shape changes between moments (subtracts each frame's coefficients from the next frame's, along the time axis)                                     
 
 
    # Summarises the whole clip. How much each number varied, its typical low and high, and how quickly it moved.
    return np.concatenate([
        C.std(axis=1),                    
        np.percentile(C, 10, axis=1),     
        np.percentile(C, 90, axis=1),     
        np.abs(d).mean(axis=1),           
        d.std(axis=1),                   
    ]).astype(np.float32)           
 
def make_feat(i, training=False, augment=None):
    """
    Turns clip i into the 100-number feature vector the SVM uses: cuts a 3-second window,
    degrades it with augmentation (training only), then makes a spectrogram and summarises it.
    """
    y, sr = cache[i]
    w, _ = norm.random_clipping(y, sr)
    w = np.ascontiguousarray(w.astype(np.float32))
    if training and augment is not None:
        w = np.ascontiguousarray(augment(samples=w, sample_rate=SR).astype(np.float32))
    return clip_features(mel.generate_mel_spectrogram(mel.bandpass_filter(w)))

# Split by recording
groups = [re.sub(r"-\d+\.wav$", "", os.path.basename(f)) for f in files]
uniq = sorted(set(groups)); random.Random(SEED).shuffle(uniq)
val_groups = set(uniq[: max(1, len(uniq) // 5)])
tr_idx = [i for i, g in enumerate(groups) if g not in val_groups]
va_idx = [i for i, g in enumerate(groups) if g in val_groups]
print(f"train {len(tr_idx)} clips | val {len(va_idx)} clips | {len(uniq)} recordings, no overlap")

N_AUG = 8 # Makes 8 copies of every training clip where each is like degraded in different ways. So like it could be quiterm noisier, muffled, distorted
try:
    from audiomentations import (AddGaussianSNR, ClippingDistortion, Compose,
                                 Gain, LowPassFilter, SevenBandParametricEQ)
    augment = Compose([
        SevenBandParametricEQ(min_gain_db=-8.0, max_gain_db=8.0, p=0.8),
        LowPassFilter(min_cutoff_freq=5000, max_cutoff_freq=11000, p=0.5),
        AddGaussianSNR(min_snr_db=3.0, max_snr_db=30.0, p=0.8),
        Gain(min_gain_db=-12.0, max_gain_db=6.0, p=0.9),
        ClippingDistortion(min_percentile_threshold=0, max_percentile_threshold=10, p=0.15),
    ])
except Exception as e:
    print(f"audiomentations unavailable ({type(e).__name__}); training without waveform augmentation")
    augment = None
 
print(f"building features ({N_AUG} augmented copies per training clip)...")
X_tr = np.stack([make_feat(i, True, augment) for _ in range(N_AUG) for i in tr_idx])
y_tr = np.tile(labels[tr_idx], N_AUG)
X_va = np.stack([make_feat(i) for i in va_idx])
y_va = labels[va_idx]
print("feature matrix:", X_tr.shape, "->", X_va.shape)

train 328 clips | val 106 clips | 74 recordings, no overlap
audiomentations unavailable (TypeError); training without waveform augmentation
building features (8 augmented copies per training clip)...
feature matrix: (2624, 100) -> (106, 100)


In [5]:
# Trains an SVM (draws a curved boundary between "call" and "not call" in feature space).
# Features are rescaled first so no single number dominates. Scored with PR-AUC on the
# held-out recordings; the baseline is what random guessing would get.

import numpy as np
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score

svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"),
)
svm.fit(X_tr, y_tr)

s = svm.decision_function(X_va)
print(f"[sklearn RBF SVM] PR-AUC {average_precision_score(y_va, s):.4f} | "
      f"ROC-AUC {roc_auc_score(y_va, s):.4f}")
print(f"baseline (always-positive) PR-AUC = {y_va.mean():.4f}")
print(f"support vectors: {svm[-1].n_support_} of {len(y_tr)} training points")

[sklearn RBF SVM] PR-AUC 0.8582 | ROC-AUC 0.9037
baseline (always-positive) PR-AUC = 0.2925
support vectors: [164 173] of 2624 training points


In [6]:
# Shows the model's mistakes: calls it scored lowest (missed calls) and non-calls
# it scored highest (false alarms), plus the average score per recording, to spot
# which recordings or sound types cause trouble.

groups_tr = np.array([groups[i] for i in tr_idx])     # recording ID per training clip
groups_aug = np.tile(groups_tr, N_AUG)                

grid = GridSearchCV(
    make_pipeline(StandardScaler(), SVC(kernel="rbf", class_weight="balanced")),
    {"svc__C": [0.1, 1, 10, 100], "svc__gamma": ["scale", 1e-3, 1e-2, 1e-1]},
    scoring="average_precision",       
    cv=GroupKFold(n_splits=4),
    n_jobs=-1,
)
grid.fit(X_tr, y_tr, groups=groups_aug)
print("best:", grid.best_params_, f"| CV PR-AUC {grid.best_score_:.4f}")

s = grid.best_estimator_.decision_function(X_va)
print(f"[tuned sklearn SVM] PR-AUC {average_precision_score(y_va, s):.4f}")

best: {'svc__C': 10, 'svc__gamma': 0.001} | CV PR-AUC 0.9879
[tuned sklearn SVM] PR-AUC 0.9152


In [ ]:
# Saves the tuned SVM only if it beats the score of the model already saved.
import os, joblib, sklearn, hashlib
from sklearn.metrics import average_precision_score

PATH = os.path.join("models", "svm_capuchin.joblib")
os.makedirs("models", exist_ok=True) 
best = grid.best_estimator_
new_score = average_precision_score(y_va, best.decision_function(X_va))   # val PR-AUC

# Fingerprint of the validation set, so we only compare scores from the same "exam".
val_id = hashlib.md5("".join(sorted(files[i] for i in va_idx)).encode()).hexdigest()

old_score, old_val_id = -1.0, None
if os.path.exists(PATH):
    old = joblib.load(PATH)
    old_score, old_val_id = old.get("val_pr_auc", -1.0), old.get("val_id")

if old_val_id is not None and old_val_id != val_id:
    print("Validation set has changed since the last save, so the scores aren't comparable. Not saving.")
elif new_score > old_score:
    joblib.dump({
        "model": best,
        "params": grid.best_params_,
        "val_pr_auc": new_score,
        "cv_pr_auc": grid.best_score_,
        "val_id": val_id,
        "n_mfcc": N_MFCC, "sr": SR, "window_size": WINDOW_SIZE,
        "sklearn_version": sklearn.__version__,
    }, PATH)
    print(f"Improved {old_score:.4f} -> {new_score:.4f}. Saved {PATH}")
else:
    print(f"No improvement ({new_score:.4f} vs saved {old_score:.4f}). Kept the old model.")

Improved -1.0000 -> 0.9152. Saved models\svm_capuchin.joblib
